# GloveSpeak — Gesture Recognition Training
## SmartGlove BISINDO Realtime Recognition
Improved with: sliding window · delta features · adaptive segmenter · per-category window

## 0. All Imports & Setup

In [ ]:
# Standard libraries
import os
import json
import warnings
from pathlib import Path
from typing import Dict, List, Tuple
from datetime import datetime

# Data processing
import numpy as np
import pandas as pd

# Machine Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress warnings
warnings.filterwarnings('ignore')

# Configuration
print("TensorFlow version:", tf.__version__)
print("GPU available:", len(tf.config.list_physical_devices('GPU')) > 0)
print("\n✅ All imports successful!")

## 1. Load Gesture List

In [ ]:
def load_gesture_list(filename='bisindo_gesture_list.txt'):
    """
    Load gesture list dari file.
    Format: CATEGORY,gesture_label
    Return: list gesture labels, dict category->indices
    """
    gestures = []
    categories = {}
    with open(filename, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split(',', 1)
            if len(parts) == 2:
                cat, label = parts[0].strip(), parts[1].strip()
                idx = len(gestures)
                gestures.append(label)
                categories.setdefault(cat, []).append(idx)
    return gestures, categories

gestures, categories = load_gesture_list()
num_gestures = len(gestures)

print(f"Total gestures: {num_gestures}")
for cat, idxs in categories.items():
    print(f"  {cat}: {len(idxs)} gestures")

print("\n📋 Sample gestures:")
for cat, idxs in list(categories.items())[:3]:
    print(f"  {cat}: {gestures[idxs[0]]}")

## 2. Load Data (per-category subfolder)

In [ ]:
def load_data_from_folders(base_path='datashet', gestures=None, categories=None):
    """
    Load semua CSV dari struktur folder:
      datashet/
        angka/    ← kategori lowercase
        huruf/
        kata/
        frasa/

    Setiap CSV: kolom timestamp + 22 sensor kolom + repetition.
    Return: X_raw (list of np.array (T,22)), y (list of int label idx)
    """
    X_raw, y = [], []

    # Buat mapping label -> index
    label_to_idx = {g: i for i, g in enumerate(gestures)}

    # Scan setiap kategori
    for cat, cat_idxs in categories.items():
        cat_dir = os.path.join(base_path, cat.lower())
        if not os.path.exists(cat_dir):
            print(f"  [WARN] Folder tidak ditemukan: {cat_dir}")
            continue

        cat_count = 0
        for csv_file in Path(cat_dir).glob('*.csv'):
            try:
                df = pd.read_csv(csv_file)

                # Kolom sensor: semua kecuali timestamp dan repetition
                drop_cols = [c for c in ['timestamp', 'repetition'] if c in df.columns]
                sensor_df = df.drop(columns=drop_cols)

                if sensor_df.shape[1] != 22:
                    continue

                data = sensor_df.values.astype(np.float32)
                if len(data) == 0:
                    continue

                # Tentukan label dari nama file
                fname = csv_file.stem
                if '_rep' in fname:
                    raw_label = fname[:fname.index('_rep')].replace('_', ' ')
                else:
                    raw_label = fname.replace('_', ' ')

                if raw_label in label_to_idx:
                    label_idx = label_to_idx[raw_label]
                else:
                    matched = [g for g in gestures if g in raw_label or raw_label in g]
                    if matched:
                        label_idx = label_to_idx[matched[0]]
                    else:
                        continue

                X_raw.append(data)
                y.append(label_idx)
                cat_count += 1

            except Exception as e:
                continue

        print(f"  {cat}: {cat_count} recordings")

    print(f"\nTotal: {len(X_raw)} recordings, {len(set(y))} unique gestures")
    return X_raw, y

X_raw, y_raw = load_data_from_folders('datashet', gestures, categories)

if len(X_raw) == 0:
    print("\n[ERROR] Tidak ada data ditemukan!")
else:
    seq_lengths = [len(s) for s in X_raw]
    print(f"\nSequence lengths: min={min(seq_lengths)}, max={max(seq_lengths)}, median={np.median(seq_lengths):.0f}")
    y_arr = np.array(y_raw)
    counts = np.bincount(y_arr, minlength=num_gestures)
    print(f"Rata-rata sampel per gesture: {counts[counts>0].mean():.1f}")

## 3. Preprocessing & Feature Engineering

In [ ]:
# Import preprocessing components
try:
    from advanced_gesture_recognition import (
        GloveSensorPreprocessor,
        CATEGORY_WINDOW,
        NUM_TOTAL_FEATURES,
        SAMPLING_RATE,
        CONFIDENCE_THRESHOLD,
    )
    print("✅ Preprocessor components loaded")
    print(f"   Window sizes: {CATEGORY_WINDOW}")
    print(f"   Total features: {NUM_TOTAL_FEATURES}")
except ImportError as e:
    print(f"⚠️  Warning: {e}")
    # Define defaults if not found
    CATEGORY_WINDOW = {'ALL': 80}
    NUM_TOTAL_FEATURES = 66
    SAMPLING_RATE = 100
    CONFIDENCE_THRESHOLD = 0.72
    print("   Using default values")

# Preprocessing
WINDOW_SIZE = CATEGORY_WINDOW.get('ALL', 80)

print(f"\nPreprocessing {len(X_raw)} sequences...")
print(f"Window size: {WINDOW_SIZE} frames")
print(f"Output features: {NUM_TOTAL_FEATURES} (raw 22 + delta 22 + accel 22)")

try:
    preprocessor = GloveSensorPreprocessor()
    preprocessor.fit(X_raw)
    X_processed = preprocessor.batch_transform(X_raw, WINDOW_SIZE)
    y_processed = np.array(y_raw)
    print(f"✅ Preprocessing complete")
    print(f"   Shape: {X_processed.shape}")
except Exception as e:
    print(f"⚠️  Preprocessing error: {e}")
    # Fallback: simple padding
    def simple_preprocess(sequences, window_size=80, num_features=22):
        processed = []
        for seq in sequences:
            if len(seq) < window_size:
                padded = np.pad(seq, ((0, window_size - len(seq)), (0, 0)), mode='constant')
            else:
                padded = seq[:window_size]
            processed.append(padded)
        return np.array(processed)
    
    X_processed = simple_preprocess(X_raw, WINDOW_SIZE, 22)
    y_processed = np.array(y_raw)
    NUM_TOTAL_FEATURES = 22
    print(f"   Shape: {X_processed.shape} [fallback preprocessing]")

## 4. Train/Validation Split & Data Augmentation

In [ ]:
# Split data
X_train, X_val, y_train, y_val = train_test_split(
    X_processed, y_processed,
    test_size=0.2, random_state=42, stratify=y_processed
)

print(f"Train: {X_train.shape} | Val: {X_val.shape}")
print(f"Label unik train: {len(np.unique(y_train))} | val: {len(np.unique(y_val))}")

# Try augmentation if available
try:
    from sensor_augmentation import SensorDataAugmenter
    
    augmenter = SensorDataAugmenter(seed=42)
    X_train_aug, y_train_aug = augmenter.augment_balanced(
        X_train, y_train,
        target_per_class=45,
        verbose=False,
    )
    print(f"\nAfter augmentation: {X_train_aug.shape}")
    X_train, y_train = X_train_aug, y_train_aug
except ImportError:
    print("⚠️  Augmentation module not found, skipping")

## 5. Build Model

In [ ]:
# Try to import model builder, otherwise build simple LSTM
try:
    from advanced_gesture_recognition import build_bilstm_attention_model, AttentionLayer
    
    model = build_bilstm_attention_model(
        num_gestures=num_gestures,
        window_size=WINDOW_SIZE,
        num_features=NUM_TOTAL_FEATURES,
        lstm_units=128,
        dense_units=128,
        dropout_rate=0.35,
    )
    print("✅ BiLSTM + Attention model loaded")
except ImportError:
    # Fallback: Simple LSTM model
    print("⚠️  Building simple LSTM (advanced model not available)")
    model = keras.Sequential([
        layers.Input(shape=(WINDOW_SIZE, NUM_TOTAL_FEATURES)),
        layers.LSTM(128, return_sequences=True),
        layers.Dropout(0.3),
        layers.LSTM(64),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_gestures, activation='softmax')
    ])

model.summary()
print(f"\nTotal parameters: {model.count_params():,}")

## 6. Compile & Train

In [ ]:
# Compile
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0005, clipnorm=1.0),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

# Callbacks
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=20,
        restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=7,
        min_lr=1e-7, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        'best_gesture_model.keras',
        monitor='val_accuracy', save_best_only=True, verbose=0
    ),
]

# Train
print(f"Training on {len(X_train)} samples for {num_gestures} gestures...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=120,
    batch_size=32,
    callbacks=callbacks,
    verbose=1,
)
print("\n✅ Training complete!")

## 7. Evaluate Model

In [ ]:
# Evaluate
train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)

print(f"Train accuracy: {train_acc*100:.2f}% (loss: {train_loss:.4f})")
print(f"Val accuracy:   {val_acc*100:.2f}% (loss: {val_loss:.4f})")

# Per-gesture accuracy
y_pred_probs = model.predict(X_val, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_confidence = np.max(y_pred_probs, axis=1)

print("\n" + "="*70)
print("Per-Gesture Accuracy:")
print("="*70)

per_gesture_acc = {}
for gesture_idx in range(num_gestures):
    mask = y_val == gesture_idx
    if mask.sum() > 0:
        acc = (y_pred[mask] == gesture_idx).mean()
        per_gesture_acc[gestures[gesture_idx]] = acc
        
        if acc < 0.7:
            print(f"  ⚠️  {gestures[gesture_idx]:25s} {acc*100:6.1f}%")

print(f"\nAverage per-gesture accuracy: {np.mean(list(per_gesture_acc.values()))*100:.1f}%")

## 8. Save Model & Metadata

In [ ]:
# Save model
model.save('best_gesture_model.keras')
print("✅ Model saved: best_gesture_model.keras")

# Save metadata
metadata = {
    'window_size': int(WINDOW_SIZE),
    'num_gestures': int(num_gestures),
    'num_features': int(NUM_TOTAL_FEATURES),
    'sampling_rate': int(SAMPLING_RATE),
    'confidence_threshold': float(CONFIDENCE_THRESHOLD),
    'gesture_labels': gestures,
    'category_windows': CATEGORY_WINDOW,
    'training_date': datetime.now().isoformat(),
}

with open('model_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
print("✅ Metadata saved: model_metadata.json")

# Try TFLite conversion
try:
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS,
        tf.lite.OpsSet.SELECT_TF_OPS
    ]
    tflite_model = converter.convert()
    with open('gesture_model.tflite', 'wb') as f:
        f.write(tflite_model)
    print(f"✅ TFLite model saved: gesture_model.tflite ({len(tflite_model)/1024:.1f} KB)")
except Exception as e:
    print(f"⚠️  TFLite conversion failed: {e}")

## 9. Test & Visualize

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set(title='Accuracy', xlabel='Epoch', ylabel='Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Val')
axes[1].set(title='Loss', xlabel='Epoch', ylabel='Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=100)
plt.show()

print("✅ Training history saved: training_history.png")

## 10. Classification Report

In [ ]:
# Classification report
present_labels = sorted(np.unique(np.concatenate([y_val, y_pred])))
target_names = [gestures[i] for i in present_labels]

print("\n" + "="*70)
print("Classification Report:")
print("="*70)
print(classification_report(y_val, y_pred, labels=present_labels, target_names=target_names, digits=3))

# Summary statistics
print("\n" + "="*70)
print("Summary:")
print("="*70)
accuracy_list = list(per_gesture_acc.values())
print(f"Best:   {sorted(per_gesture_acc.items(), key=lambda x: x[1])[-1]}")
print(f"Worst:  {sorted(per_gesture_acc.items(), key=lambda x: x[1])[0]}")
print(f"Mean:   {np.mean(accuracy_list)*100:.1f}%")
print(f"Median: {np.median(accuracy_list)*100:.1f}%")
print("="*70)

## 11. Ready for Android Deployment ✅

### Files generated:
- `best_gesture_model.keras` — Keras model (full precision)
- `gesture_model.tflite` — TensorFlow Lite model (Android)
- `model_metadata.json` — Configuration & gesture labels
- `training_history.png` — Training curves

### Next steps:
1. Copy model files to Android project
2. Use TensorFlow Lite Interpreter for inference
3. Apply preprocessing using metadata parameters
4. Handle real-time streaming from ESP32